# R5: Build the final tables

**Goal:** turn the R3 and R4 flags into decisions, and build the final tables for the rest of the thesis:
the master table v2 (everything kept), the analysis-ready table v2 (rules applied), a cleaning report,
a comparison with the old v1 table, and a data dictionary with the source of every range.
Input: `data/processed_v2/R4_attributes.gpkg`.

## Step 1: Load the R4 table and list the flags

**Found:** 1,207 fields, 83 columns, 26 flag columns (14 from R3 and 12 from R4).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

here = Path.cwd()
ROOT = here if (here / "data" / "raw").exists() else here.parent
DATA2 = ROOT / "data" / "processed_v2"
OUT = ROOT / "results" / "phase1a"

f = gpd.read_file(DATA2 / "R4_attributes.gpkg")
FLAGS = [c for c in f.columns if c.startswith("flag_")]
for c in FLAGS:
    f[c] = f[c].fillna(False).astype(bool)
assert len(f) == 1207 and f["uid"].is_unique, "input is not the full R4 table"

print("fields:", len(f), "| columns:", f.shape[1], "| flag columns:", len(FLAGS))
print(pd.Series({c: int(f[c].sum()) for c in FLAGS}).to_string())

## Step 2: Decide what to do with each flag

Rule: a field is removed only when it cannot be used at all. Everything else is kept with its flag,
because the yield checks in Phase 1B decide the rest ("how do you know?" needs evidence, not an assumption).
Two fields are removed: the field with no polygon (no location, so no satellite data), and one field of
the exact duplicate pair (same yield, variety, dates and irrigation; the first by field ID is kept, so no
label changes). The taluka is taken from the real boundary; the recorded one is kept as `taluka_recorded`.

**Found:** 2 fields removed (1 no polygon, 1 duplicate), 1,205 remain. 46 talukas were replaced by the
boundary taluka. Every flag has a written action, saved in `R5_rules.csv`.

In [ ]:
# keep the raw taluka, and use the taluka from the real boundary (R3 Step 4) as the working one
f["taluka_recorded"] = f["taluka"]
f["taluka"] = f["taluka_boundary"].fillna(f["taluka"])

# the two removal rules (everything else is kept, with its flag)
f["drop_reason"] = ""
f.loc[f["flag_no_polygon"], "drop_reason"] = "no polygon"
dup_groups = f[f["flag_duplicate_record"]].sort_values("uid").groupby("overlap_group")
for g, rows in dup_groups:
    f.loc[rows.index[1:], "drop_reason"] = "duplicate record (first by field ID kept)"
f["in_analysis"] = f["drop_reason"] == ""

# rule table: every flag, the action taken, and how many fields it touched
ACTION = {
    "flag_no_polygon": "removed (no location, so no satellite data)",
    "flag_duplicate_record": "one of each pair removed (same field entered twice)",
    "flag_taluka_mismatch": "kept; taluka taken from the real boundary",
    "flag_outside_district": "kept (0.19 and 2.06 km outside: boundary precision)",
    "flag_overlap_same_year": "kept, flagged; overlap group keeps them together; yields checked in Phase 1B",
    "flag_area_mismatch": "kept; area comes from the polygon",
    "flag_provider_area_negative": "kept; provider area not used",
    "flag_geom_repaired": "fixed in R3 (make_valid)", "flag_repair_changed_area": "fixed in R3; flagged",
    "flag_slivers_removed": "fixed in R3", "flag_split_field": "kept (real split field)",
    "flag_tiny": "none found", "flag_huge": "none found",
    "flag_repeated_plot": "kept; tag used to group the folds (Phase 5)",
    "flag_sowing_year_fixed": "fixed in R4", "flag_sowing_implausible": "kept; sowing date left blank",
    "flag_sowing_missing": "kept, flagged; yields checked in Phase 1B",
    "flag_sowing_unusual": "kept, flagged",
    "flag_harvest_year_fixed": "fixed in R4", "flag_harvest_swapped": "fixed in R4",
    "flag_harvest_undecided": "kept; harvest date left blank",
    "flag_duration_unusual": "kept, flagged",
    "flag_harvest_before_maturity": "kept, flagged; yields checked in Phase 1B",
    "flag_variety_typo": "fixed in R4", "flag_variety_unknown": "kept; variety = Unknown",
    "flag_soil_zero_blanked": "fixed in R4",
}
missing_rule = [c for c in FLAGS if c not in ACTION]
assert not missing_rule, f"flags with no rule: {missing_rule}"
rules = pd.DataFrame({"flag": FLAGS,
                      "fields": [int(f[c].sum()) for c in FLAGS],
                      "of_these_removed": [int((f[c] & ~f["in_analysis"]).sum()) for c in FLAGS],
                      "action": [ACTION[c] for c in FLAGS]})
rules.to_csv(OUT / "R5_rules.csv", index=False)
print(rules.to_string(index=False))
print("\nremoved:", f.loc[~f["in_analysis"], "drop_reason"].value_counts().to_dict())
print("fields in the analysis-ready table:", int(f["in_analysis"].sum()))
print("taluka changed to the boundary taluka:", int((f["taluka"] != f["taluka_recorded"]).sum()))

## Step 3: Save the master and analysis-ready tables

The master table keeps all 1,207 fields, every value and every flag, plus `in_analysis` and `drop_reason`.
The analysis-ready table keeps the 1,205 fields, without the helper columns used inside the checks.

**Found:** master v2 has 1,207 fields and 86 columns. Analysis-ready v2 has 1,205 fields and 75 columns
(55, 60, 323 and 767 by year). 338 of them carry at least one "kept, flagged" doubt, to be checked in Phase 1B.

In [ ]:
# helper columns from the checks: kept in the master table, left out of the analysis-ready one
HELPERS = ["geom_valid_raw", "area_m2_before", "area_m2_buffer0", "area_m2_after", "repair_area_change_pct",
           "n_parts_raw", "sliver_area_m2", "taluka_gadm", "variety_match"]

master = f.copy()
master.to_file(DATA2 / "yield_master_v2.gpkg", driver="GPKG")

ready = f[f["in_analysis"]].drop(columns=[c for c in HELPERS + ["drop_reason", "in_analysis"] if c in f.columns])
assert ready["uid"].is_unique and ready.geometry.notna().all()
ready.to_file(DATA2 / "yield_analysis_ready_v2.gpkg", driver="GPKG")

print(f"master v2: {len(master)} fields, {master.shape[1]} columns")
print(f"analysis-ready v2: {len(ready)} fields, {ready.shape[1]} columns")
print("analysis-ready by year:", ready["year"].value_counts().sort_index().to_dict())
KEPT_FLAGGED = [c for c in FLAGS if ACTION[c].startswith("kept") and c != "flag_repeated_plot"]
print("fields kept with at least one doubt flag:", int(ready[KEPT_FLAGGED].any(axis=1).sum()))

## Step 4: Cleaning report

A flow of field counts from the raw files to the analysis-ready table, and a figure of how many fields
each check touched, coloured by what was done (removed, fixed, kept with a flag, or a tag).

**Found:** the count stays at 1,207 through R1 to R4 (nothing is deleted while checking) and drops to 1,205
in R5. Most checks fixed or flagged values; only 2 fields were removed.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# flow of field counts, raw to analysis-ready
flow = pd.DataFrame([
    {"step": "Raw files (R1)", "fields": 1207},
    {"step": "One table, personal data removed (R2)", "fields": 1207},
    {"step": "Geometry checked, nothing removed (R3)", "fields": 1207},
    {"step": "Attributes checked, nothing removed (R4)", "fields": 1207},
    {"step": "No polygon removed (R5)", "fields": int(1207 - (f["drop_reason"] == "no polygon").sum())},
    {"step": "Duplicate record removed (R5)", "fields": int(f["in_analysis"].sum())},
])
flow.to_csv(OUT / "R5_cleaning_flow.csv", index=False)
print(flow.to_string(index=False))

# figure: fields touched by each check, coloured by what was done
FIG = ROOT / "reports" / "figures" / "phase1a"; FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.family": "Times New Roman", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
r = rules[rules["fields"] > 0].copy()
r["kind"] = np.where(r["flag"].isin(["flag_no_polygon", "flag_duplicate_record"]), "removed", np.where(r["action"].str.startswith("fixed"), "fixed", "kept, flagged"))
r.loc[r["flag"] == "flag_repeated_plot", "kind"] = "tag (not a problem)"
r = r.sort_values("fields")
colors = {"removed": "#C00000", "fixed": "#3F7CAC", "kept, flagged": "#E0A458", "tag (not a problem)": "#BFBFBF"}
fig, ax = plt.subplots(figsize=(7.5, 6.2))
ax.barh(r["flag"].str.replace("flag_", "").str.replace("_", " "), r["fields"], color=r["kind"].map(colors))
for y, v in enumerate(r["fields"]):
    ax.text(v + 2, y, str(v), va="center", fontsize=9.5)
ax.set_xlabel("Fields")
ax.legend(handles=[Patch(color=c, label=k) for k, c in colors.items()], frameon=False, loc="lower right")
ax.grid(axis="x", color="#E5E5E5", lw=0.6); ax.set_axisbelow(True)
fig.tight_layout(); fig.savefig(FIG / "R5_checks_overview.png", dpi=300, facecolor="white"); plt.show()

## Step 5: Compare with v1

The old v1 table (notebook 07) is compared field by field with v2.

**Found:** v1 has 1,204 fields, v2 has 1,205. 2 fields are in v1 only (the no-polygon field and the duplicate).
3 fields are in v2 only: yields of 0, 260 and 720, which v1 removed with fixed thresholds; v2 leaves yield
decisions to Phase 1B. Of the 1,202 fields in both, 107 harvest dates changed (the day/month fix that v1 missed),
46 talukas, 3 sowing dates, 2 varieties (752 and 3344, guessed in v1) and 1 area (the repaired polygon).

In [ ]:
# compare with the old v1 analysis-ready table
v1 = gpd.read_file(next((ROOT / "data").rglob("yield_analysis_ready.gpkg")))
print(f"v1: {len(v1)} fields, {v1.shape[1]} columns | v2: {len(ready)} fields, {ready.shape[1]} columns")

only_v1, only_v2 = set(v1["uid"]) - set(ready["uid"]), set(ready["uid"]) - set(v1["uid"])
why = f.set_index("uid")
print("in v1 only:", len(only_v1), why.loc[list(only_v1), "drop_reason"].value_counts().to_dict() if only_v1 else "")
print("in v2 only:", len(only_v2), "| their yields:", sorted(why.loc[list(only_v2), "crop_yield"].tolist()),
      "(v1 removed them by fixed yield thresholds; v2 leaves yield to Phase 1B)")

# fields in both: which values changed?
both = v1.set_index("uid").join(ready.set_index("uid"), lsuffix="_v1", rsuffix="_v2", how="inner")
def changed(a, b):
    return ~((a == b) | (a.isna() & b.isna()))
comp = pd.DataFrame([
    {"value": "sowing date", "changed": int(changed(both["sowing_date_v1"].dt.normalize(), both["sowing_date_v2"].dt.normalize()).sum())},
    {"value": "harvest date", "changed": int(changed(both["harvest_date_v1"].dt.normalize(), both["harvest_date_v2"].dt.normalize()).sum())},
    {"value": "variety", "changed": int(changed(both["variety_v1"].astype(str).str.upper().str.replace(" ", ""),
                                                 both["variety_v2"].astype(str).str.upper().str.replace(" ", "")).sum())},
    {"value": "taluka", "changed": int(changed(both["taluka_v1"], both["taluka_v2"]).sum())},
    {"value": "area (more than 1% different)", "changed": int(((both["area_acres_v2"] / both["area_acres_v1"] - 1).abs() > 0.01).sum())},
])
comp["of"] = len(both)
comp.to_csv(OUT / "R5_compare_v1.csv", index=False)
print(f"\nfields in both: {len(both)}")
print(comp.to_string(index=False))

## Step 6: Data dictionary v2

Each main column with its meaning, unit, sensible range and the source of that range (owed since R1).
Every flag is listed with the action taken.

**Found:** 51 rows (25 value columns and 26 flags). Saved as `R5_data_dictionary_v2.csv`.

In [ ]:
# data dictionary v2: meaning, unit, sensible range and where that range comes from
D = [
 ("uid", "field ID from the survey", "-", "unique", "R1: no duplicates"),
 ("year", "kharif season", "year", "2022 to 2025", "raw files"),
 ("batch", "survey batch (A, B, or 2025)", "-", "A, B, 2025", "R2 Step 1"),
 ("farmer_code", "anonymous farmer code (sha1 of name and phone)", "-", "-", "R2; replaces personal data"),
 ("survey_code", "anonymous code of village + land survey number", "-", "-", "R2; replaces personal data"),
 ("taluka", "taluka from the real boundary", "-", "10 talukas of Latur", "DataMeet village boundaries, Census 2001 (R3 Step 4)"),
 ("taluka_recorded", "taluka as written in the survey", "-", "-", "raw files"),
 ("village", "village name", "-", "112 villages", "raw files; spelling checked in R2 Step 7"),
 ("irrigation", "irrigation type", "-", "Rainfed, Drip, Flow/Protective", "raw files"),
 ("variety", "standard variety name", "-", "11 released varieties or Unknown", "matched by code number (R4 Step 3); ICAR and IISR variety lists"),
 ("crop_yield", "yield as recorded", "unit decided in Phase 1B (q/acre or q/ha)", "decided in Phase 1B", "Phase 1B: official Latur yields and variety potential"),
 ("sowing_date", "sowing date after the checks", "date", "1 June to 5 August", "Maharashtra sowing-date trials, Kathmale et al. (R4 Step 1)"),
 ("harvest_date", "harvest date after the day/month fix", "date", "within the 3 IQR fences of its year", "R4 Step 2 (chi-square, likelihood ratio)"),
 ("crop_duration", "harvest minus sowing", "days", "at least 91 days", "earliest listed maturity, MAUS 612 (ICAR oilseeds compendium)"),
 ("area_acres", "field area from the polygon (UTM 43N)", "acres", "0.17 to 31.3 acres", "3 IQR fences on log-area (R3 Step 2); checked against EPSG:6933"),
 ("area_actual_farmer", "farmer-reported area", "acres", "within 20% of area_acres", "R3 Step 3 (unit ratio test)"),
 ("soil_ph", "soil pH (lab)", "-", "acidic < 6.5, neutral 6.5 to 7.0, alkaline > 7.0", "Soil Health Card limits, ICAR-IISS"),
 ("soil_carbon", "organic carbon (lab)", "%", "low < 0.5, high > 0.75", "Soil Health Card limits, ICAR-IISS"),
 ("soil_n", "available nitrogen (lab)", "kg/ha", "low < 280, high > 560", "Soil Health Card limits, ICAR-IISS"),
 ("soil_p", "available phosphorus (lab)", "kg/ha", "low < 10, high > 25", "Soil Health Card limits, ICAR-IISS"),
 ("soil_k", "available potassium (lab)", "kg/ha", "low < 120, high > 280", "Soil Health Card limits, ICAR-IISS"),
 ("soil_ec", "electrical conductivity (lab)", "dS/m", "saline > 4", "Soil Health Card limits, ICAR-IISS"),
 ("overlap_group", "fields drawn on the same land in one year", "-", "-", "R3 Step 5 (share of smaller field > 10%)"),
 ("plot_id", "same land in different years", "-", "-", "R3 Step 6 (share of smaller field > 0.5, sensitivity tested)"),
 ("*_group", "category with rare groups merged", "-", "at least 10 fields per group", "Cochran (1954) rule (R4 Step 5)"),
]
dic = pd.DataFrame(D, columns=["column", "meaning", "unit", "sensible range", "source of the range"])
flags = pd.DataFrame({"column": FLAGS, "meaning": "True/False check result", "unit": "-",
                      "sensible range": "-", "source of the range": [ACTION[c] for c in FLAGS]})
dic = pd.concat([dic, flags], ignore_index=True)
dic.to_csv(OUT / "R5_data_dictionary_v2.csv", index=False)
print(f"data dictionary v2: {len(dic)} rows ({len(D)} value columns, {len(flags)} flags)")
print(dic.head(14).to_string(index=False))

## Summary of R5

**Phase 1A is complete.** From the four raw files to one checked dataset, with every step counted.

1. **Rules:** only 2 fields removed (no polygon, duplicate record). All other doubtful fields are kept with
   their flag, so Phase 1B can test them with the yield values.
2. **Tables:** `yield_master_v2.gpkg` (1,207 fields, all flags) and `yield_analysis_ready_v2.gpkg`
   (1,205 fields), in `data/processed_v2/`.
3. **Cleaning report:** the flow of field counts and a figure of every check.
4. **Against v1:** 107 harvest dates, 46 talukas and 2 varieties differ; 3 extreme yields are back in,
   for Phase 1B to judge with statistics.
5. **Data dictionary v2:** every range now has a source.

**Saved in `results/phase1a/`:** R5_rules, R5_cleaning_flow, R5_compare_v1, R5_data_dictionary_v2.
**Figure:** `reports/figures/phase1a/R5_checks_overview.png`.

**Next: Phase 1B,** the yield labels: units, statistics and outliers.